# Fake News Prediction with the Saved Best Model
This notebook is fully independent of the training notebook. It loads **only** `models/best_model/` (the best model + its RAG retriever / preprocessing objects), applies the identical preprocessing and retrieval-augmentation pipeline, and predicts whether a news statement is **Fake** or **Real** with a confidence score. No training happens here.

## 1. Import Required Libraries
Imports the libraries needed for preprocessing, retrieval and inference.

In [1]:
import os, re, json
import numpy as np
import pandas as pd
import joblib
import torch
os.environ.setdefault("NLTK_ALLOW_PROXIED_URLOPEN", "1")
import nltk
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings; warnings.filterwarnings("ignore")

## 2. Load NLTK Resources
Makes sure WordNet and stop-words (used for polysemy glosses) are available.

In [2]:
for res in ["wordnet", "omw-1.4", "stopwords"]:
    nltk.download(res, quiet=True)
from nltk.corpus import wordnet as wn, stopwords
from nltk.wsd import lesk

## 3. Set Paths
Points to the saved best-model folder only.

In [3]:
BEST_DIR = os.path.abspath(os.path.join("..", "models", "best_model"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(BEST_DIR); print(sorted(os.listdir(BEST_DIR)))

/home/claude/PROJECT_MAIN/models/best_model
['best_model_info.json', 'model', 'rag_retriever']


## 4. Load Best-Model Information
Reads which model was selected and its stored evaluation metrics.

In [4]:
info = json.load(open(os.path.join(BEST_DIR, "best_model_info.json")))
LABEL_MAP = {int(k): v for k, v in info["label_map"].items()}
THRESHOLD = info["threshold"]
print("Best model :", info["best_model"], f"({info['type']})")
print("Test metrics:", {k: round(v, 4) for k, v in info["test_metrics"].items()})

Best model : roberta_base (transformer)
Test metrics: {'accuracy': 0.6314, 'precision': 0.5661, 'recall': 0.6815, 'f1': 0.6185, 'macro_f1': 0.631, 'roc_auc': 0.6681}


## 5. Define Preprocessing and RAG Functions
Same cleaning, retrieval and word-sense functions used during training.

In [5]:
STOP_WORDS = set(stopwords.words("english"))

def clean_text(text):
    """Normalise quotes/whitespace exactly as done during training."""
    t = str(text)
    t = (t.replace("“", '"').replace("”", '"')
           .replace("‘", "'").replace("’", "'"))
    t = re.sub(r"\s+", " ", t).strip()
    return t

def truncate_words(text, n):
    w = text.split()
    return " ".join(w[:n])

def prepare_statement(text, cfg):
    """Cleaned statement used as model input (very long outliers are capped)."""
    return truncate_words(clean_text(text), cfg["max_statement_words"])

def retrieve_neighbors(query_texts, vectorizer, kb_matrix, k, exclude_idx=None, chunk=1024):
    """TF-IDF cosine retrieval over the fact-checked knowledge base.
    exclude_idx[i] = KB row of query i itself (leave-one-out for training rows), or -1."""
    q = vectorizer.transform(query_texts)
    all_idx, all_sim = [], []
    for s in range(0, q.shape[0], chunk):
        sims = (q[s:s + chunk] @ kb_matrix.T).toarray()
        if exclude_idx is not None:
            for r, ex in enumerate(exclude_idx[s:s + chunk]):
                if ex >= 0:
                    sims[r, ex] = -1.0
        top = np.argsort(-sims, axis=1)[:, :k]
        all_idx.append(top)
        all_sim.append(np.take_along_axis(sims, top, axis=1))
    return np.vstack(all_idx), np.vstack(all_sim)

def polysemy_glosses(text, context_tokens, cfg):
    """Pick the most polysemous content words (WordNet) and disambiguate them with
    Lesk using the statement + retrieved evidence as context (RAG-enhanced WSD)."""
    words, seen = [], set()
    for w in re.findall(r"[A-Za-z]+", text.lower()):
        if w in STOP_WORDS or len(w) < 3 or w in seen:
            continue
        seen.add(w)
        n = len(wn.synsets(w))
        if n >= cfg["min_senses"]:
            words.append((n, w))
    words.sort(key=lambda x: (-x[0], x[1]))
    out = []
    for n, w in words[: cfg["max_polysemous_words"]]:
        sense = lesk(context_tokens, w)
        if sense is not None:
            out.append(f"{w}: {truncate_words(sense.definition(), cfg['max_gloss_words'])}")
    return out

def build_rag_context(text, nb_idx, nb_sim, kb_texts, kb_labels, cfg):
    """Second input segment for the RAG models: retrieved fact-checked claims
    (with their verdicts) + WordNet senses of polysemous words."""
    evid, ctx_tokens = [], re.findall(r"[A-Za-z]+", text.lower())
    for j, s in zip(nb_idx, nb_sim):
        if s < cfg["min_similarity"]:
            continue
        nt = truncate_words(kb_texts[j], cfg["max_evidence_words"])
        evid.append(f"[{cfg['label_tags'][int(kb_labels[j])]}] {nt}")
        ctx_tokens += re.findall(r"[A-Za-z]+", nt.lower())
    gl = polysemy_glosses(text, ctx_tokens, cfg)
    parts = ["Evidence: " + (" | ".join(evid) if evid else "none")]
    parts.append("Senses: " + ("; ".join(gl) if gl else "none"))
    return " ".join(parts)

def make_rag_inputs(texts, vectorizer, kb_matrix, kb_texts, kb_labels, cfg, exclude_idx=None):
    idx, sim = retrieve_neighbors(texts, vectorizer, kb_matrix, cfg["top_k"], exclude_idx)
    return [build_rag_context(t, idx[i], sim[i], kb_texts, kb_labels, cfg) for i, t in enumerate(texts)]

## 6. Define Model Loading and Inference Functions
Rebuilds a transformer from its config + `model.pth` and returns P(Fake).

In [6]:
def load_text_classifier(model_dir, device="cpu"):
    """Rebuild architecture from config.json, load PyTorch weights (model.pth) + tokenizer."""
    config = AutoConfig.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_config(config)
    state = torch.load(os.path.join(model_dir, "model.pth"), map_location=device, mmap=True)
    model.load_state_dict(state, assign=True)  # assign + mmap avoids holding two copies in RAM
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    return model.to(device).eval(), tokenizer

@torch.no_grad()
def predict_proba_transformer(model, tokenizer, texts_a, texts_b=None, max_len=128, batch_size=64):
    """Return P(class=1 'fake') for each input, using the same tokenisation as training."""
    model.eval()
    probs = []
    for s in range(0, len(texts_a), batch_size):
        a = list(texts_a[s:s + batch_size])
        b = None if texts_b is None else list(texts_b[s:s + batch_size])
        enc = tokenizer(a, b, truncation="longest_first" if b is not None else True,
                        max_length=max_len, padding=True, return_tensors="pt")
        enc.pop("token_type_ids", None)
        logits = model(**{k: v.to(model.device) for k, v in enc.items()}).logits
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs) if probs else np.array([])

## 7. Load the RAG Retriever and Knowledge Base
Loads the fitted TF-IDF retriever, the fact-checked knowledge base and the RAG configuration.

In [7]:
RET_DIR = os.path.join(BEST_DIR, "rag_retriever")
retriever_vec = joblib.load(os.path.join(RET_DIR, "tfidf_retriever.joblib"))
kb = pd.read_csv(os.path.join(RET_DIR, "knowledge_base.csv"))
kb_texts, kb_labels = kb["text"].astype(str).tolist(), kb["label"].to_numpy()
kb_matrix = retriever_vec.transform(kb_texts)
RAG_CFG = json.load(open(os.path.join(RET_DIR, "rag_config.json")))
RAG_CFG["label_tags"] = {int(k): v for k, v in RAG_CFG["label_tags"].items()}
print("Knowledge base:", kb_matrix.shape, "| RAG config:", RAG_CFG)

Knowledge base: (7153, 60424) | RAG config: {'top_k': 3, 'min_similarity': 0.05, 'max_evidence_words': 15, 'max_statement_words': 60, 'max_polysemous_words': 2, 'min_senses': 4, 'max_gloss_words': 8, 'label_tags': {0: 'real', 1: 'fake'}}


## 8. Load the Best Model
Loads the saved best model (for an ensemble: its saved member models and weights).

In [8]:
def load_component(path):
    meta_file = os.path.join(path, "metadata.json")
    meta = json.load(open(meta_file))
    if meta["framework"] == "scikit-learn":
        return {"kind": "sklearn", "model": joblib.load(os.path.join(path, "model.joblib")), "meta": meta}
    m, t = load_text_classifier(path, DEVICE)
    return {"kind": "transformer", "model": m, "tokenizer": t, "meta": meta}

if info["type"] == "ensemble":
    ens_cfg = json.load(open(os.path.join(BEST_DIR, "ensemble_config.json")))
    components = {n: load_component(os.path.join(BEST_DIR, "members", n)) for n in ens_cfg["weights"]}
    weights = ens_cfg["weights"]
else:
    components = {info["best_model"]: load_component(os.path.join(BEST_DIR, "model"))}
    weights = {info["best_model"]: 1.0}
for n, c in components.items():
    print(f"loaded {n:24s} kind={c['kind']:12s} uses_rag={c['meta']['uses_rag']} weight={weights[n]}")

loaded roberta_base             kind=transformer  uses_rag=False weight=1.0


## 9. Define the Prediction Pipeline
Cleans the input, builds the RAG context (retrieval + word senses), runs the model(s) and returns label + confidence.

In [9]:
def predict_news(statements):
    statements = [s for s in statements if str(s).strip()]
    stmts = [prepare_statement(s, RAG_CFG) for s in statements]
    clean = [clean_text(s) for s in statements]
    contexts = make_rag_inputs(clean, retriever_vec, kb_matrix, kb_texts, kb_labels, RAG_CFG)
    p_fake = np.zeros(len(stmts))
    for n, c in components.items():
        if c["kind"] == "sklearn":
            p = c["model"].predict_proba(stmts)[:, 1]
        else:
            p = predict_proba_transformer(c["model"], c["tokenizer"], stmts,
                                          contexts if c["meta"]["uses_rag"] else None, c["meta"]["max_len"])
        p_fake += weights[n] * p
    pred = (p_fake >= THRESHOLD).astype(int)
    return pd.DataFrame({"statement": statements, "prediction": [LABEL_MAP[i] for i in pred],
                         "p_fake": p_fake.round(4), "confidence": np.where(pred == 1, p_fake, 1 - p_fake).round(4),
                         "rag_context": contexts})

## 10. Sample Inputs
Example news statements for testing (new, user-style inputs).

In [10]:
sample_news = [
    "Says the federal government will confiscate all privately owned guns starting next month.",
    "The unemployment rate dropped to its lowest level in a decade last year.",
    "Says a new vaccine contains microchips that track people's location.",
    "Our state spends more on prisons than on higher education.",
]

## 11. Run Predictions on Sample Inputs
Generates predictions with probability and confidence.

In [11]:
results = predict_news(sample_news)
pd.set_option("display.max_colwidth", 120)
results[["statement", "prediction", "p_fake", "confidence"]]

,statement,prediction,p_fake,confidence
0,Says the federal government will confiscate all privately owned guns starting next month.,Fake,0.5539,0.5539
1,The unemployment rate dropped to its lowest level in a decade last year.,Real,0.4146,0.5854
2,Says a new vaccine contains microchips that track people's location.,Fake,0.5466,0.5466
3,Our state spends more on prisons than on higher education.,Real,0.4018,0.5982


## 12. Show Retrieved Evidence for One Input
Displays the retrieved claims + word senses for the first sample (fed to the model only if it is a RAG model).

In [12]:
print("Model uses RAG context:", any(c["meta"]["uses_rag"] for c in components.values()))
print("STATEMENT :", results.statement[0])
print("PREDICTION:", results.prediction[0], f"(confidence {results.confidence[0]:.2%})")
print("CONTEXT   :", results.rag_context[0])

Model uses RAG context: False
STATEMENT : Says the federal government will confiscate all privately owned guns starting next month.
PREDICTION: Fake (confidence 55.39%)
CONTEXT   : Evidence: [real] Point Pleasant Beach does not own its beaches; they are privately owned and operated by | [fake] Says the federal government is earning more tax revenue now than any other time. | [fake] Says the federal government restricts how much salt we can put on our food. Senses: starting: begin an event that is implied and limited; says: the chance to speak


## 13. Predict Your Own Statement
Edit `user_input` to classify any news statement or headline.

In [13]:
user_input = "Says the governor cut education funding by 20 percent while raising his own salary."
r = predict_news([user_input]).iloc[0]
print(f"Input      : {r.statement}")
print(f"Prediction : {r.prediction.upper()} NEWS")
print(f"P(Fake)    : {r.p_fake:.4f}")
print(f"Confidence : {r.confidence:.2%}")

Input      : Says the governor cut education funding by 20 percent while raising his own salary.
Prediction : REAL NEWS
P(Fake)    : 0.4351
Confidence : 56.49%
